## 02 · Profil ile ilan karşılaştırması (ön kontrol)
Ajanın değerlendirmesinden önce, standart kayıtlar üzerinden hangi koşulların **uyuştuğunu**, hangilerinin **uyuşmadığını** ve hangilerinin **ilanda belirtilmediğini** ayırıyorum.

In [1]:
# Ortam hazırlığı: Colab'da repo klonlanır, yerelde notebooks/ klasöründen proje köküne geçilir
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("job-opportunity-mcp-agent").exists() and not Path("README.md").exists():
    !git clone -q https://github.com/alimdemir/job-opportunity-mcp-agent.git
if IN_COLAB and Path("job-opportunity-mcp-agent").exists():
    %cd job-opportunity-mcp-agent
    !pip -q install -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("çalışma klasörü:", Path.cwd().name)

çalışma klasörü: 02-job-opportunity-mcp-agent


In [2]:
import re
import pandas as pd
from jobs import load_posts, normalize

profile = {
    "skills": {"Python", "PostgreSQL", "Docker", "Kubernetes", "AWS"},
    "learning": {"Rust", "Terraform"},          # öğrenmek istediği, deneyim sayılmaz
    "work_mode": "remote",
    "can_work_from": "Türkiye",                  # ABD/Kanada ile sınırlı ilanlar uymaz
}

In [3]:
def evaluate(job):
    stack = set(job.tech_stack)
    matched = sorted(stack & profile["skills"])
    notes = []
    if job.work_mode is None:
        notes.append("çalışma biçimi belirtilmemiş")
    elif job.work_mode != profile["work_mode"]:
        notes.append(f"{job.work_mode} - uzaktan değil")
    if job.location and re.search(r"\b(US|USA|Canada)\b", job.location) and job.work_mode == "remote":
        notes.append("uzaktan ama bölge kısıtlı")
    if not stack:
        notes.append("teknoloji bilgisi yok")
    return {"şirket": job.company, "eşleşen beceri": ", ".join(matched) or "-",
            "yalnızca öğrenme hedefi": ", ".join(sorted(stack & profile["learning"])) or "-",
            "not": "; ".join(notes) or "ön kontrolden geçti"}

pd.set_option("display.max_colwidth", 60)
pd.DataFrame([evaluate(normalize(p)) for p in load_posts()])

,şirket,eşleşen beceri,yalnızca öğrenme hedefi,not
0,Snout,"AWS, PostgreSQL, Python",-,uzaktan ama bölge kısıtlı
1,Flywheel Motion,-,-,ön kontrolden geçti
2,Open Education Applications / Neon,"Kubernetes, PostgreSQL, Python",Terraform,hybrid - uzaktan değil
3,PostHog,-,-,teknoloji bilgisi yok
4,SmarterDx,-,-,uzaktan ama bölge kısıtlı; teknoloji bilgisi yok
5,Shepherd,-,-,onsite - uzaktan değil
6,Senzing,"AWS, Kubernetes, PostgreSQL, Python","Rust, Terraform",uzaktan ama bölge kısıtlı
7,Kyra Health,Python,-,uzaktan ama bölge kısıtlı
8,Marple,"AWS, Docker, Kubernetes, PostgreSQL, Python",-,hybrid - uzaktan değil
9,Palace Cybersecurity,-,-,uzaktan ama bölge kısıtlı
